# NBA Next-Season Salary Prediction (Class-Style Neural Network)

This notebook is designed to match your in-class neural network style:
- `StandardScaler` preprocessing
- Keras `Sequential`/Functional MLP patterns
- `EarlyStopping` + `ReduceLROnPlateau`
- optional Optuna tuning

It is built for your NBA cleaned dataset and keeps the project framing **performance-only** (no salary-history predictors).


## Modeling Setup

- Target: `next_log_salary` (predict next-season salary in log space)
- Split: train (older seasons), validation (2021-2022), holdout (2023-2025)
- Feature pack: `prior_perf_rolling_v1` from your existing performance-model pipeline
- Leakage control: excludes salary-history predictors (`salary`, `prev_salary`, etc.)


In [1]:
import os
import sys
import types
import random
from pathlib import Path

# ---- Environment stability workaround ----
# In this local setup, importing real pyarrow can crash the TF kernel.
# This lightweight stub keeps sklearn checks happy without loading pyarrow binary.
fake_pa = types.ModuleType("pyarrow")
fake_pa.__version__ = "0.0.0"
for _n in ["Table", "RecordBatch", "Array", "ChunkedArray"]:
    setattr(fake_pa, _n, type(_n, (), {}))
sys.modules["pyarrow"] = fake_pa

import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)


TensorFlow: 2.21.0
Pandas: 2.3.2


In [2]:
# Paths and project imports
PROJECT_ROOT = Path('/Users/amritdhillon/Desktop/Advanced ML/Final Project')
NBA_DATA_DIR = PROJECT_ROOT / 'NBA Data'
PERF_MODEL_DIR = NBA_DATA_DIR / 'PositionModelsPreseasonPerformanceV1'
CSV_PATH = NBA_DATA_DIR / 'NBADataCleanV4.csv'

assert CSV_PATH.exists(), f"Missing CSV: {CSV_PATH}"

if str(PERF_MODEL_DIR) not in sys.path:
    sys.path.append(str(PERF_MODEL_DIR))

from model_common_preseason_performance_v1 import (
    _ensure_engineered_columns,
    _feature_pack_map,
    _position_mask,
)

print("CSV path:", CSV_PATH)


CSV path: /Users/amritdhillon/Desktop/Advanced ML/Final Project/NBA Data/NBADataCleanV4.csv


In [3]:
# Load and engineer dataset using existing project feature logic
raw_df = pd.read_csv(CSV_PATH)
df = _ensure_engineered_columns(raw_df)

print("Raw rows:", len(raw_df))
print("Engineered rows (targetable):", len(df))
print("Columns:", len(df.columns))


Raw rows: 10469
Engineered rows (targetable): 8336
Columns: 165


In [4]:
# Configuration
TRAIN_GAMES_THRESHOLD = 15
HOLDOUT_START_SEASON = 2023
VAL_START_SEASON = 2021
VAL_END_SEASON = 2022

FEATURE_PACK = 'prior_perf_rolling_v1'   # stronger than core pack in most runs
POSITION_CODE = None                      # None=ALL, or 'PG','SG','SF','PF','C'

RUN_OPTUNA = False   # set True if you want hyperparameter tuning
OPTUNA_TRIALS = 20

# non-monetary categorical context
CAT_FEATURES_EXTRA = [
    'position',
    'position_primary_group_5',
    'archetype_label',
    'cba_service_bucket',
    'team_region',
]

# explicit blocklist for leakage (salary-history related)
LEAKAGE_BLOCKLIST = {
    'salary',
    'prev_salary',
    'prev2_salary',
    'prev_salary_growth',
    'salary_volatility_3yr',
    'big_raise_last_year',
    'big_cut_last_year',
    'next_salary',
    'next_log_salary',
}


In [5]:
def metrics_dict(y_true_log, pred_log):
    y_true_log = np.asarray(y_true_log).reshape(-1)
    pred_log = np.asarray(pred_log).reshape(-1)

    # guardrail against exploding outputs
    pred_log = np.clip(pred_log, np.log(300_000), np.log(80_000_000))

    rmse_log = float(np.sqrt(mean_squared_error(y_true_log, pred_log)))
    mae_log = float(mean_absolute_error(y_true_log, pred_log))
    r2_log = float(r2_score(y_true_log, pred_log))

    y_true_usd = np.exp(y_true_log)
    pred_usd = np.exp(pred_log)

    rmse_usd = float(np.sqrt(mean_squared_error(y_true_usd, pred_usd)))
    mae_usd = float(mean_absolute_error(y_true_usd, pred_usd))
    r2_usd = float(r2_score(y_true_usd, pred_usd))

    return {
        'r2_log': r2_log,
        'rmse_log': rmse_log,
        'mae_log': mae_log,
        'r2_usd': r2_usd,
        'rmse_usd': rmse_usd,
        'mae_usd': mae_usd,
    }


def print_metrics(title, m):
    print(f"\n{title}")
    print("R2 (log):", round(m['r2_log'], 6))
    print("RMSE (log):", round(m['rmse_log'], 6))
    print("MAE (log):", round(m['mae_log'], 6))
    print("R2 (USD):", round(m['r2_usd'], 6))
    print("RMSE (USD):", round(m['rmse_usd'], 2))
    print("MAE (USD):", round(m['mae_usd'], 2))


In [6]:
def build_time_split_dataset(df_in, position_code=None, feature_pack='prior_perf_rolling_v1'):
    data = df_in.copy()

    if position_code is not None:
        data = data[_position_mask(data['position'], position_code)].copy()

    pack_map = _feature_pack_map(position_code)
    num_features = [
        c for c in pack_map[feature_pack]
        if c in data.columns and c not in LEAKAGE_BLOCKLIST
    ]

    cat_features = [c for c in CAT_FEATURES_EXTRA if c in data.columns]

    eligible_train = (data['games'] >= TRAIN_GAMES_THRESHOLD) & data['next_log_salary'].notna()
    holdout_mask = (data['season'] >= HOLDOUT_START_SEASON) & data['next_log_salary'].notna()

    train_pool = data[eligible_train & (data['season'] <= HOLDOUT_START_SEASON - 1)].copy()
    holdout_df = data[holdout_mask].copy()

    val_mask = train_pool['season'].between(VAL_START_SEASON, VAL_END_SEASON)
    train_df = train_pool[~val_mask].copy()
    val_df = train_pool[val_mask].copy()

    # fallback if small validation slice for a position
    if len(val_df) < 40:
        fallback_n = max(40, int(0.15 * len(train_pool)))
        val_df = train_pool.tail(fallback_n).copy()
        train_df = train_pool.iloc[:-fallback_n].copy()

    info = {
        'position': position_code if position_code is not None else 'ALL',
        'feature_pack': feature_pack,
        'n_numeric_features': len(num_features),
        'n_cat_features': len(cat_features),
        'train_rows': len(train_df),
        'val_rows': len(val_df),
        'holdout_rows': len(holdout_df),
    }

    return train_df, val_df, holdout_df, num_features, cat_features, info


In [7]:
def fit_preprocessors(train_df, num_features, cat_features):
    num_imputer = SimpleImputer(strategy='median')
    num_scaler = StandardScaler()

    Xn_train_raw = train_df[num_features].replace([np.inf, -np.inf], np.nan)
    Xn_train_imp = num_imputer.fit_transform(Xn_train_raw)
    Xn_train = num_scaler.fit_transform(Xn_train_imp)

    if len(cat_features) > 0:
        try:
            ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=5)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
        Xc_train = ohe.fit_transform(train_df[cat_features].fillna('UNK').astype(str))
    else:
        ohe = None
        Xc_train = np.empty((len(train_df), 0), dtype='float32')

    return num_imputer, num_scaler, ohe, Xn_train, Xc_train


def transform_features(df_part, num_features, cat_features, num_imputer, num_scaler, ohe):
    Xn_raw = df_part[num_features].replace([np.inf, -np.inf], np.nan)
    Xn = num_scaler.transform(num_imputer.transform(Xn_raw))

    if ohe is not None and len(cat_features) > 0:
        Xc = ohe.transform(df_part[cat_features].fillna('UNK').astype(str))
    else:
        Xc = np.empty((len(df_part), 0), dtype='float32')

    X = np.hstack([Xn, Xc]).astype('float32')
    y_log = df_part['next_log_salary'].astype('float32').values
    return X, y_log


In [8]:
# Build dataset matrices
train_df, val_df, holdout_df, num_features, cat_features, info = build_time_split_dataset(
    df,
    position_code=POSITION_CODE,
    feature_pack=FEATURE_PACK,
)

num_imputer, num_scaler, ohe, _, _ = fit_preprocessors(train_df, num_features, cat_features)

X_train, y_train_log = transform_features(train_df, num_features, cat_features, num_imputer, num_scaler, ohe)
X_val, y_val_log = transform_features(val_df, num_features, cat_features, num_imputer, num_scaler, ohe)
X_hold, y_hold_log = transform_features(holdout_df, num_features, cat_features, num_imputer, num_scaler, ohe)

# Standardize target for training stability
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_log.reshape(-1, 1)).ravel().astype('float32')
y_val = y_scaler.transform(y_val_log.reshape(-1, 1)).ravel().astype('float32')

print(info)
print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_hold shape:', X_hold.shape)


{'position': 'ALL', 'feature_pack': 'prior_perf_rolling_v1', 'n_numeric_features': 79, 'n_cat_features': 5, 'train_rows': 6445, 'val_rows': 747, 'holdout_rows': 758}
X_train shape: (6445, 113)
X_val shape: (747, 113)
X_hold shape: (758, 113)


In [9]:
# Class-style baseline model (similar structure to in-class MLPs)
def build_baseline_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='linear')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.Huber(delta=1.0),
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model


def build_regularized_mlp(input_dim, h1=256, h2=128, h3=64, dropout=0.2, l2=1e-4, lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(h1, activation='relu', kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),

        layers.Dense(h2, activation='relu', kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),

        layers.Dense(h3, activation='relu', kernel_regularizer=regularizers.l2(l2)),
        layers.Dropout(dropout * 0.8),

        layers.Dense(1, activation='linear'),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss=keras.losses.Huber(delta=0.75),
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model


def train_model(model, X_train, y_train, X_val, y_val, epochs=300, batch_size=64, verbose=0):
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
        callbacks=callbacks,
    )
    return history


def predict_log(model, X, y_scaler):
    y_hat_scaled = model.predict(X, verbose=0).reshape(-1)
    y_hat_log = y_scaler.inverse_transform(y_hat_scaled.reshape(-1, 1)).reshape(-1)
    return y_hat_log


In [10]:
# Train baseline and regularized models
baseline = build_baseline_mlp(X_train.shape[1])
hist_baseline = train_model(baseline, X_train, y_train, X_val, y_val, epochs=220, batch_size=64, verbose=0)

reg_model = build_regularized_mlp(X_train.shape[1], h1=320, h2=192, h3=96, dropout=0.18, l2=1.5e-4, lr=8e-4)
hist_reg = train_model(reg_model, X_train, y_train, X_val, y_val, epochs=320, batch_size=64, verbose=0)

# Evaluate
pred_val_base = predict_log(baseline, X_val, y_scaler)
pred_hold_base = predict_log(baseline, X_hold, y_scaler)

pred_val_reg = predict_log(reg_model, X_val, y_scaler)
pred_hold_reg = predict_log(reg_model, X_hold, y_scaler)

m_val_base = metrics_dict(y_val_log, pred_val_base)
m_hold_base = metrics_dict(y_hold_log, pred_hold_base)

m_val_reg = metrics_dict(y_val_log, pred_val_reg)
m_hold_reg = metrics_dict(y_hold_log, pred_hold_reg)

print('Model info:', info)
print_metrics('Baseline Validation', m_val_base)
print_metrics('Baseline Holdout', m_hold_base)
print_metrics('Regularized Validation', m_val_reg)
print_metrics('Regularized Holdout', m_hold_reg)


Model info: {'position': 'ALL', 'feature_pack': 'prior_perf_rolling_v1', 'n_numeric_features': 79, 'n_cat_features': 5, 'train_rows': 6445, 'val_rows': 747, 'holdout_rows': 758}

Baseline Validation
R2 (log): 0.701463
RMSE (log): 0.704694
MAE (log): 0.510509
R2 (USD): 0.763213
RMSE (USD): 5848334.87
MAE (USD): 3629029.75

Baseline Holdout
R2 (log): 0.641646
RMSE (log): 0.83442
MAE (log): 0.572237
R2 (USD): 0.782881
RMSE (USD): 5901795.65
MAE (USD): 3902315.38

Regularized Validation
R2 (log): 0.683887
RMSE (log): 0.725141
MAE (log): 0.525319
R2 (USD): 0.742975
RMSE (USD): 6093135.35
MAE (USD): 3760313.44

Regularized Holdout
R2 (log): 0.533865
RMSE (log): 0.951667
MAE (log): 0.626626
R2 (USD): 0.738546
RMSE (USD): 6476387.1
MAE (USD): 4333834.35


In [11]:
# Optional hyperparameter tuning (class-style extension inspired by your Optuna in-class work)
best_params = {
    'h1': 320,
    'h2': 192,
    'h3': 96,
    'dropout': 0.18,
    'l2': 1.5e-4,
    'lr': 8e-4,
    'batch_size': 64,
}

if RUN_OPTUNA:
    try:
        import optuna

        def objective(trial):
            params = {
                'h1': trial.suggest_int('h1', 128, 512, step=64),
                'h2': trial.suggest_int('h2', 64, 256, step=32),
                'h3': trial.suggest_int('h3', 32, 128, step=32),
                'dropout': trial.suggest_float('dropout', 0.05, 0.35),
                'l2': trial.suggest_float('l2', 1e-6, 5e-4, log=True),
                'lr': trial.suggest_float('lr', 2e-4, 2e-3, log=True),
                'batch_size': trial.suggest_categorical('batch_size', [32, 64, 96, 128]),
            }

            model = build_regularized_mlp(
                input_dim=X_train.shape[1],
                h1=params['h1'],
                h2=params['h2'],
                h3=params['h3'],
                dropout=params['dropout'],
                l2=params['l2'],
                lr=params['lr'],
            )

            _ = train_model(
                model,
                X_train,
                y_train,
                X_val,
                y_val,
                epochs=220,
                batch_size=params['batch_size'],
                verbose=0,
            )

            val_pred = predict_log(model, X_val, y_scaler)
            val_mae_usd = metrics_dict(y_val_log, val_pred)['mae_usd']
            return val_mae_usd

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=OPTUNA_TRIALS)

        best_params = study.best_params
        print('Optuna best params:', best_params)
        print('Optuna best objective (val MAE USD):', study.best_value)
    except Exception as e:
        print('Optuna skipped due to error:', e)

best_params


{'h1': 320,
 'h2': 192,
 'h3': 96,
 'dropout': 0.18,
 'l2': 0.00015,
 'lr': 0.0008,
 'batch_size': 64}

In [12]:
# Train final model with best params and evaluate holdout
final_model = build_regularized_mlp(
    input_dim=X_train.shape[1],
    h1=int(best_params['h1']),
    h2=int(best_params['h2']),
    h3=int(best_params['h3']),
    dropout=float(best_params['dropout']),
    l2=float(best_params['l2']),
    lr=float(best_params['lr']),
)

_ = train_model(
    final_model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=320,
    batch_size=int(best_params['batch_size']),
    verbose=0,
)

final_val_pred = predict_log(final_model, X_val, y_scaler)
final_hold_pred = predict_log(final_model, X_hold, y_scaler)

final_val_metrics = metrics_dict(y_val_log, final_val_pred)
final_hold_metrics = metrics_dict(y_hold_log, final_hold_pred)

print_metrics('Final Validation', final_val_metrics)
print_metrics('Final Holdout', final_hold_metrics)



Final Validation
R2 (log): 0.661337
RMSE (log): 0.750559
MAE (log): 0.529529
R2 (USD): 0.776192
RMSE (USD): 5685798.43
MAE (USD): 3643657.79

Final Holdout
R2 (log): 0.578076
RMSE (log): 0.905411
MAE (log): 0.613361
R2 (USD): 0.730086
RMSE (USD): 6580339.12
MAE (USD): 4420250.52


In [13]:
# Position-level evaluation using the same class-style tuned architecture
positions = [None, 'PG', 'SG', 'PF', 'SF', 'C']
position_results = []

for pos in positions:
    tr, va, ho, nf, cf, inf = build_time_split_dataset(df, position_code=pos, feature_pack=FEATURE_PACK)

    ni, ns, enc, _, _ = fit_preprocessors(tr, nf, cf)
    Xtr, ytr_log = transform_features(tr, nf, cf, ni, ns, enc)
    Xva, yva_log = transform_features(va, nf, cf, ni, ns, enc)
    Xho, yho_log = transform_features(ho, nf, cf, ni, ns, enc)

    ys = StandardScaler()
    ytr = ys.fit_transform(ytr_log.reshape(-1, 1)).ravel().astype('float32')
    yva = ys.transform(yva_log.reshape(-1, 1)).ravel().astype('float32')

    mdl = build_regularized_mlp(
        input_dim=Xtr.shape[1],
        h1=int(best_params['h1']),
        h2=int(best_params['h2']),
        h3=int(best_params['h3']),
        dropout=float(best_params['dropout']),
        l2=float(best_params['l2']),
        lr=float(best_params['lr']),
    )

    _ = train_model(
        mdl, Xtr, ytr, Xva, yva,
        epochs=260,
        batch_size=int(best_params['batch_size']),
        verbose=0,
    )

    hold_pred = predict_log(mdl, Xho, ys)
    hold_m = metrics_dict(yho_log, hold_pred)

    position_results.append({
        'position': inf['position'],
        'feature_pack': FEATURE_PACK,
        'rows_holdout': inf['holdout_rows'],
        'mae_usd_holdout': hold_m['mae_usd'],
        'rmse_usd_holdout': hold_m['rmse_usd'],
        'r2_usd_holdout': hold_m['r2_usd'],
        'mae_log_holdout': hold_m['mae_log'],
    })

position_summary = pd.DataFrame(position_results)
position_summary


,position,feature_pack,rows_holdout,mae_usd_holdout,rmse_usd_holdout,r2_usd_holdout,mae_log_holdout
0,ALL,prior_perf_rolling_v1,758,4.110050e+06,6.142067e+06,0.764843,0.601431
1,PG,prior_perf_rolling_v1,144,5.999568e+06,9.012875e+06,0.642030,0.746447
2,SG,prior_perf_rolling_v1,192,4.858824e+06,7.514062e+06,0.560254,0.681438
3,PF,prior_perf_rolling_v1,145,4.627362e+06,7.083718e+06,0.731413,0.674191
4,SF,prior_perf_rolling_v1,152,3.975477e+06,6.147362e+06,0.704681,0.607135
5,C,prior_perf_rolling_v1,136,4.108144e+06,6.701530e+06,0.660339,0.549088


In [14]:
# Save outputs
out_dir = PROJECT_ROOT / 'Neural Network' / 'results'
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / 'nn_classstyle_holdout_summary.csv'
position_summary.to_csv(summary_path, index=False)

final_path = out_dir / 'nn_classstyle_all_holdout_metrics.json'
with open(final_path, 'w', encoding='utf-8') as f:
    import json
    json.dump({
        'position': info['position'],
        'feature_pack': FEATURE_PACK,
        'best_params': {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in best_params.items()},
        'validation_metrics': final_val_metrics,
        'holdout_metrics': final_hold_metrics,
    }, f, indent=2)

print('Saved:', summary_path)
print('Saved:', final_path)
position_summary


Saved: /Users/amritdhillon/Desktop/Advanced ML/Final Project/Neural Network/results/nn_classstyle_holdout_summary.csv
Saved: /Users/amritdhillon/Desktop/Advanced ML/Final Project/Neural Network/results/nn_classstyle_all_holdout_metrics.json


,position,feature_pack,rows_holdout,mae_usd_holdout,rmse_usd_holdout,r2_usd_holdout,mae_log_holdout
0,ALL,prior_perf_rolling_v1,758,4.110050e+06,6.142067e+06,0.764843,0.601431
1,PG,prior_perf_rolling_v1,144,5.999568e+06,9.012875e+06,0.642030,0.746447
2,SG,prior_perf_rolling_v1,192,4.858824e+06,7.514062e+06,0.560254,0.681438
3,PF,prior_perf_rolling_v1,145,4.627362e+06,7.083718e+06,0.731413,0.674191
4,SF,prior_perf_rolling_v1,152,3.975477e+06,6.147362e+06,0.704681,0.607135
5,C,prior_perf_rolling_v1,136,4.108144e+06,6.701530e+06,0.660339,0.549088


## Why this notebook should be stronger than the prior NN notebook

1. Closer to your in-class MLP setup (scaling + dense stacks + callbacks), so training behavior is more predictable.
2. Uses stronger regularization and learning-rate scheduling for tabular stability.
3. Uses target standardization and robust loss (Huber) to reduce extreme prediction blowups.
4. Includes optional Optuna tuning to search better architecture/learning-rate/batch-size settings.
5. Keeps strict no-salary-history leakage while preserving rich performance/context features.
